# RAG Query: Before and After

Ask an LLM a question **without RAG** (vague / wrong), then **with RAG**
(accurate, cited). Run `rag_ingestion.ipynb` first to populate Milvus.

This notebook:
1. Deploys a vLLM query LLM (Granite-8B) via KServe
2. Loads the embedding model to encode queries
3. Compares LLM answers without and with RAG context

## Install dependencies

## Install dependencies

In [ ]:
%pip install -q pymilvus sentence-transformers openai torch requests

## Configure

> The embedding model **must match** the model used during ingestion
> (`ibm-granite/granite-embedding-125m-english`). Mismatched embeddings
> produce meaningless retrieval results.

In [ ]:
NAMESPACE = "rag-example"

# Milvus connection (must match ingestion settings)
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
MILVUS_DB = "default"
COLLECTION_NAME = "rag_documents"

# Must match the model used during ingestion.
EMBEDDING_MODEL = "ibm-granite/granite-embedding-125m-english"

# Query LLM — deployed via KServe in the next section.
LLM_SERVICE_NAME = "granite-8b"
MODEL_NAME = "ibm-granite/granite-3.1-8b-instruct"
TOP_K = 5

## Deploy Query LLM

Deploy Granite-8B as a KServe InferenceService using vLLM.
Skip this cell if the model is already deployed.

In [ ]:
from rag_helpers import deploy_vllm_service, test_llm_endpoint, wait_for_service

INFERENCE_URL = deploy_vllm_service(
    name=LLM_SERVICE_NAME,
    namespace=NAMESPACE,
    model_name=MODEL_NAME,
    extra_args=["--enforce-eager"],
)

wait_for_service(LLM_SERVICE_NAME, NAMESPACE)
test_llm_endpoint(INFERENCE_URL, MODEL_NAME)

## Initialize clients

In [ ]:
from openai import OpenAI
from pymilvus import MilvusClient
from rag_helpers import ask_llm, build_context, print_comparison, search_milvus
from sentence_transformers import SentenceTransformer

milvus = MilvusClient(uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB)
# sentence-transformers is used here for simplicity; if ingestion used vLLM
# embeddings, minor floating-point differences are negligible for retrieval.
embed_model = SentenceTransformer(EMBEDDING_MODEL)
llm = OpenAI(base_url=f"{INFERENCE_URL}/v1", api_key="unused")

milvus.load_collection(COLLECTION_NAME)
stats = milvus.get_collection_stats(COLLECTION_NAME)
row_count = stats.get("row_count", stats.get("data", {}).get("row_count", "?"))
print(f"Collection '{COLLECTION_NAME}': {row_count} vectors")

## Pick a question

In [ ]:
# Change this to a question relevant to the PDFs you ingested.
# If you used the "Download Sample PDFs" cell in rag_ingestion.ipynb,
# the papers cover language models, retrieval-augmented generation, and embeddings.
QUESTION = "How does retrieval-augmented generation improve the accuracy of language model responses?"
print(QUESTION)

## Query WITHOUT RAG

In [ ]:
answer_no_rag = ask_llm(QUESTION, llm=llm, model_name=MODEL_NAME)
print(answer_no_rag)

## Retrieve context from Milvus

In [ ]:
chunks = search_milvus(
    QUESTION,
    milvus=milvus,
    embed_model=embed_model,
    collection_name=COLLECTION_NAME,
    top_k=TOP_K,
)

for i, c in enumerate(chunks, 1):
    preview = c["text"][:100].replace("\n", " ")
    print(f"[{i}] {c['source_file']}  chunk {c['chunk_index']}  score {c['score']:.3f}")
    print(f"    {preview}...\n")

## Query WITH RAG

In [ ]:
context = build_context(chunks)
answer_with_rag = ask_llm(QUESTION, llm=llm, model_name=MODEL_NAME, context=context)
print(answer_with_rag)

## Compare side by side

In [ ]:
print_comparison(QUESTION, answer_no_rag, answer_with_rag, chunks)

## Cleanup (optional)

Delete the query LLM deployment when done.

In [ ]:
# Uncomment to delete the LLM deployment:
# delete_vllm_service(LLM_SERVICE_NAME, NAMESPACE)